[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/40.%20The%20Port%20Capacity%20%26%20Expansion%20Timing%20Problem/P40-Tier-1_executed.ipynb)

---

# 40. The Port Capacity and Expansion Timing Problem
## Tier 1: The Pen-and-Paper Method — Markov Decision Process Formulation

### Goal and outputs

This notebook formulates a port capacity expansion problem as a discounted Markov decision process (MDP). It uses value iteration to estimate the best expansion action for each represented capacity-and-demand state. The main outputs are the current state's selected action, its estimated discounted value, a convergence trace, and visual summaries of the model inputs and action values.

The example concerns the Port of Meridian, with an initial capacity of 3.2 million TEU and demand of 2.8 million TEU. The available actions are no expansion, a 15% minor upgrade, a 40% major expansion, and an 85% comprehensive expansion.

### Scope and modeling assumptions

- Demand grows according to one of three scenarios: conservative, baseline, or aggressive. Each scenario has a fixed growth rate and probability, and the probabilities sum to one.
- Capacity and demand are represented on finite grids. After an action and a demand-growth realization, the resulting capacity and demand are each projected to their nearest grid level. Values outside the grid are represented by its nearest boundary level. This projection makes the state space finite and is an approximation of continuous capacity and demand.
- Utilization is demand divided by capacity and is not capped at 100%. Throughput, however, cannot exceed capacity. Keeping demand above capacity visible allows the model to represent unmet demand and increasing congestion.
- Each period's action cost is charged in that period. The model does not separately record whether an earlier expansion has already been undertaken, so action costs and capacity changes are interpreted as recurring period decisions within this stylized model.
- Infrastructure age is stored as descriptive state information but does not affect rewards, transitions, or the policy. The value iteration therefore operates on the capacity-and-demand grid, not on age-dependent decisions.
- The solver uses discounted value iteration, which evaluates an infinite stream of modeled periods. It is not a finite-horizon model: there is no terminal period or terminal value.
- The model omits implementation delays, construction cost overruns, financing constraints, environmental restrictions, and strategic responses by competing ports. Its output is a result for the stated assumptions, not a complete investment appraisal.

### The MDP in plain language

A state records the port's capacity and current utilization; together these determine demand. At each state, the decision-maker chooses an expansion action. Demand growth is uncertain, so each action can lead to several possible next states. The reward for a period combines throughput revenue with investment, congestion, and maintenance costs. The MDP accounts for both the reward now and the expected value of future states.

Formally, an MDP consists of a state set, an action set, transition probabilities, a reward function, and a discount factor. Here, the state set is the discretized capacity-and-demand grid, the actions are the four expansion alternatives, and the transition probabilities come from the demand-growth scenarios.

### State, reward, and transition definitions

Let a state be \(s=(C,u,a)\), where \(C\) is capacity in million TEU, \(u\) is utilization, and \(a\) is infrastructure age in years. Demand in million TEU is \(D(s)=uC\). Let \(A\) be the set of expansion actions, and let scenario \(k\) have probability \(p_k\) and growth rate \(g_k\). The next capacity and demand before grid projection are

$$
C' = C(1+r_a), \qquad D' = D(s)(1+g_k), \qquad u' = \frac{D'}{C'},
$$

where \(r_a\) is the action's proportional capacity increase. The state used by the solver is obtained by projecting \(C'\) and \(D'\) separately to their nearest represented grid levels; its utilization is the projected demand divided by the projected capacity.

For throughput \(Q(s)=\min(D(s),C)\), the immediate reward, in millions of dollars per period, is

$$
R(s,a) = 180Q(s) - c_a - \frac{365P(u)}{10^6} - 0.02C,
$$

where \(c_a\) is the action cost in millions of dollars, \(P(u)\) is the daily congestion penalty in dollars, and \(0.02C\) is the annual maintenance charge in millions of dollars under the code's stated rate convention. The revenue term follows because throughput is in million TEU and revenue is 180 dollars per TEU. The congestion penalty is zero at or below utilization 0.8 and grows exponentially above that threshold.

### Bellman equation and value iteration

The value \(V(s)\) is the maximum expected discounted sum of future rewards starting from state \(s\). The Bellman optimality equation is

$$
V(s) = \max_{a \in A} \left[ R(s,a) + \gamma \sum_k p_k V\bigl(T(s,a,g_k)\bigr) \right],
$$

where \(T(s,a,g_k)\) is the projected next state and \(\gamma=0.92\) is the discount factor. Value iteration repeatedly applies the Bellman update, starting from zero:

$$
V_0(s)=0, \qquad
V_{n+1}(s)=\max_{a \in A}\left[R(s,a)+\gamma\sum_k p_kV_n\bigl(T(s,a,g_k)\bigr)\right].
$$

The iterations stop when the largest absolute change across states is below the specified tolerance, or when the iteration limit is reached. Since \(0\leq\gamma<1\), the Bellman update is a contraction for this finite, bounded-reward model; this supports convergence to the discounted infinite-horizon value function. With \(S\) states, \(A\) actions, and \(K\) scenarios, one update requires \(O(SAK)\) work. The code records the largest change at each iteration and then extracts a maximizing action for every state. If actions tie, the first action in the defined action order is selected.

In the implementation, `CapacityState` represents a state, `ExpansionAction` represents an action, `calculate_reward` implements \(R\), and `get_next_state` implements the transition before grid projection. `solve_mdp_value_iteration` performs the Bellman updates and extracts the policy. The discounted value is not a finite-horizon project NPV: it is the model's value under the repeated-period reward and decision assumptions above.

### How to read the results

The selected action for the initial state is the action with the greatest estimated Bellman value under the scenario probabilities, reward assumptions, discount factor, and state-grid projection. The action comparison plot shows the one-step reward plus discounted continuation value for each action at that state. The convergence plot shows the maximum value-function update at each iteration. Neither plot establishes that the model's assumptions are realistic; interpretation should remain conditional on the stated model.

In [ ]:
# These libraries support the MDP model, numerical calculations, and visual summaries.
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import Dict, List

plt.style.use("default")
sns.set_palette("husl")

print("Libraries imported successfully for MDP formulation.")

In [ ]:
@dataclass
class CapacityState:
    """Represent capacity, utilization, and descriptive infrastructure age."""

    capacity_mteu: float
    utilization_rate: float
    infrastructure_age: float

    def get_congestion_penalty(self) -> float:
        """Return the daily congestion penalty in dollars."""
        alpha = 1000
        beta = 2.0
        threshold = 0.8

        if self.utilization_rate <= threshold:
            return 0.0
        return alpha * np.exp(beta * (self.utilization_rate - threshold))


@dataclass
class ExpansionAction:
    """Represent a capacity decision and its period investment cost."""

    name: str
    capacity_increase: float
    cost_millions: float


class PortCapacityMDP:
    """Define the discretized port capacity expansion MDP."""

    def __init__(self):
        self.revenue_per_teu = 180
        self.discount_factor = 0.92
        self.maintenance_cost_rate = 0.02

        self.actions = {
            0: ExpansionAction("No expansion", 0.0, 0),
            1: ExpansionAction("Minor upgrade", 0.15, 120),
            2: ExpansionAction("Major expansion", 0.40, 450),
            3: ExpansionAction("Comprehensive", 0.85, 750),
        }

        self.demand_scenarios = {
            "conservative": {"growth_rate": 0.04, "probability": 0.35},
            "baseline": {"growth_rate": 0.07, "probability": 0.45},
            "aggressive": {"growth_rate": 0.105, "probability": 0.20},
        }

        probability_sum = sum(
            scenario["probability"]
            for scenario in self.demand_scenarios.values()
        )
        if not np.isclose(probability_sum, 1.0):
            raise ValueError("Demand scenario probabilities must sum to one.")

        self.states = self._initialize_states()
        self.value_function = {}
        self.optimal_policy = {}

    def _initialize_states(self) -> List[CapacityState]:
        """Create the initial state and the capacity-demand grid."""
        states = []

        current_demand = 2.8
        current_capacity = 3.2
        states.append(
            CapacityState(
                capacity_mteu=current_capacity,
                utilization_rate=current_demand / current_capacity,
                infrastructure_age=0,
            )
        )

        capacity_levels = [3.2, 3.68, 4.48, 5.92]
        demand_levels = [2.8, 3.0, 3.5, 4.0]

        for capacity in capacity_levels:
            for demand in demand_levels:
                states.append(
                    CapacityState(
                        capacity_mteu=capacity,
                        utilization_rate=demand / capacity,
                        infrastructure_age=0,
                    )
                )

        return states

    def get_transition_probabilities(
        self, state: CapacityState, action: ExpansionAction
    ) -> Dict[str, Dict[str, float]]:
        """Return the demand-growth scenarios used for every state-action pair."""
        return self.demand_scenarios

    def calculate_reward(
        self, state: CapacityState, action: ExpansionAction
    ) -> float:
        """Calculate the immediate reward in millions of dollars per period."""
        demand = state.utilization_rate * state.capacity_mteu
        throughput = min(demand, state.capacity_mteu)

        # Million TEU multiplied by dollars per TEU gives millions of dollars.
        revenue_millions = throughput * self.revenue_per_teu
        investment_cost_millions = action.cost_millions
        congestion_cost_millions = (
            state.get_congestion_penalty() * 365 / 1_000_000
        )
        maintenance_cost_millions = (
            state.capacity_mteu * self.maintenance_cost_rate
        )

        return (
            revenue_millions
            - investment_cost_millions
            - congestion_cost_millions
            - maintenance_cost_millions
        )

    def get_next_state(
        self,
        state: CapacityState,
        action: ExpansionAction,
        demand_growth: float,
    ) -> CapacityState:
        """Apply an action and demand growth before projecting to the state grid."""
        new_capacity = state.capacity_mteu * (1 + action.capacity_increase)
        current_demand = state.utilization_rate * state.capacity_mteu
        new_demand = current_demand * (1 + demand_growth)
        new_utilization = new_demand / new_capacity

        return CapacityState(
            capacity_mteu=new_capacity,
            utilization_rate=new_utilization,
            infrastructure_age=state.infrastructure_age + 1,
        )

    def find_nearest_state_index(self, next_state: CapacityState) -> int:
        """Project next capacity and demand separately onto the represented grid."""
        capacity_levels = sorted(
            {state.capacity_mteu for state in self.states}
        )
        demand_levels = sorted(
            {
                state.capacity_mteu * state.utilization_rate
                for state in self.states
            }
        )

        next_demand = next_state.capacity_mteu * next_state.utilization_rate
        projected_capacity = min(
            capacity_levels,
            key=lambda level: abs(level - next_state.capacity_mteu),
        )
        projected_demand = min(
            demand_levels,
            key=lambda level: abs(level - next_demand),
        )

        for index, state in enumerate(self.states):
            state_demand = state.capacity_mteu * state.utilization_rate
            if (
                np.isclose(state.capacity_mteu, projected_capacity)
                and np.isclose(state_demand, projected_demand)
            ):
                return index

        raise RuntimeError("The projected capacity-demand state is not represented.")


print("MDP classes and the discretized state representation are ready.")

In [ ]:
def solve_mdp_value_iteration(
    mdp: PortCapacityMDP,
    max_iterations: int = 1000,
    tolerance: float = 1e-6,
) -> Dict:
    """Solve the discounted MDP by synchronous value iteration."""
    mdp.value_function = {
        index: 0.0 for index in range(len(mdp.states))
    }
    mdp.optimal_policy = {}
    convergence_history = []
    converged = False

    for iteration in range(1, max_iterations + 1):
        new_value_function = {}

        # Every state in this sweep uses the same previous value function.
        for state_index, state in enumerate(mdp.states):
            action_values = []

            for action in mdp.actions.values():
                expected_future_value = 0.0

                for scenario in mdp.get_transition_probabilities(
                    state, action
                ).values():
                    next_state = mdp.get_next_state(
                        state, action, scenario["growth_rate"]
                    )
                    next_state_index = mdp.find_nearest_state_index(next_state)
                    expected_future_value += (
                        scenario["probability"]
                        * mdp.value_function[next_state_index]
                    )

                action_value = (
                    mdp.calculate_reward(state, action)
                    + mdp.discount_factor * expected_future_value
                )
                action_values.append(action_value)

            new_value_function[state_index] = max(action_values)

        max_change = max(
            abs(new_value_function[index] - mdp.value_function[index])
            for index in new_value_function
        )
        mdp.value_function = new_value_function
        convergence_history.append(max_change)

        if max_change < tolerance:
            converged = True
            break

    # Recover the maximizing action from the converged value estimates.
    for state_index, state in enumerate(mdp.states):
        action_values = []

        for action in mdp.actions.values():
            expected_future_value = 0.0

            for scenario in mdp.get_transition_probabilities(
                state, action
            ).values():
                next_state = mdp.get_next_state(
                    state, action, scenario["growth_rate"]
                )
                next_state_index = mdp.find_nearest_state_index(next_state)
                expected_future_value += (
                    scenario["probability"]
                    * mdp.value_function[next_state_index]
                )

            action_values.append(
                mdp.calculate_reward(state, action)
                + mdp.discount_factor * expected_future_value
            )

        mdp.optimal_policy[state_index] = int(np.argmax(action_values))

    return {
        "iterations": len(convergence_history),
        "converged": converged,
        "convergence_history": convergence_history,
        "final_values": mdp.value_function,
        "optimal_policy": mdp.optimal_policy,
    }


print("Value iteration solver is ready.")

In [ ]:
# Solve the MDP and inspect the policy for the initial port state.
mdp = PortCapacityMDP()
solution_results = solve_mdp_value_iteration(mdp)

print(f"Value iteration completed in {solution_results['iterations']} iterations.")
print(f"Converged to the requested tolerance: {solution_results['converged']}")
print(
    "Final maximum value-function change: "
    f"{solution_results['convergence_history'][-1]:.2e}"
)

current_state_idx = 0
current_state = mdp.states[current_state_idx]
optimal_action_id = solution_results["optimal_policy"][current_state_idx]
optimal_action = mdp.actions[optimal_action_id]
optimal_value = solution_results["final_values"][current_state_idx]

print("\n" + "=" * 60)
print("DISCOUNTED MDP DECISION FOR THE PORT OF MERIDIAN")
print("=" * 60)
print(f"Current capacity: {current_state.capacity_mteu:.1f} million TEU")
current_demand = current_state.utilization_rate * current_state.capacity_mteu
print(f"Current demand: {current_demand:.1f} million TEU")
print(f"Current utilization: {current_state.utilization_rate:.1%}")
print(
    "Daily congestion penalty: "
    f"${current_state.get_congestion_penalty():,.0f}"
)
print(f"Selected action: {optimal_action.name}")
print(f"Capacity increase: {optimal_action.capacity_increase:.0%}")
print(f"Period investment cost: ${optimal_action.cost_millions} million")
print(f"Estimated discounted value: ${optimal_value:.1f} million")

In [ ]:
# Summarize convergence, action values, congestion, and scenario inputs.
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Port Capacity Expansion MDP Analysis",
    fontsize=16,
    fontweight="bold",
)

axes[0, 0].plot(solution_results["convergence_history"])
axes[0, 0].set_title("Value Iteration Convergence")
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Maximum Value-Function Change")
axes[0, 0].set_yscale("log")
axes[0, 0].grid(True, alpha=0.3)

action_names = [action.name for action in mdp.actions.values()]
action_values = []

for action in mdp.actions.values():
    expected_future_value = 0.0

    for scenario in mdp.get_transition_probabilities(
        current_state, action
    ).values():
        next_state = mdp.get_next_state(
            current_state, action, scenario["growth_rate"]
        )
        next_state_index = mdp.find_nearest_state_index(next_state)
        expected_future_value += (
            scenario["probability"]
            * solution_results["final_values"][next_state_index]
        )

    action_values.append(
        mdp.calculate_reward(current_state, action)
        + mdp.discount_factor * expected_future_value
    )

action_colors = [
    "lightgreen" if index == optimal_action_id else "lightcoral"
    for index in range(len(action_names))
]
bars = axes[0, 1].bar(action_names, action_values, color=action_colors)
axes[0, 1].set_title("Bellman Action Values at the Initial State")
axes[0, 1].set_ylabel("Discounted Value (million dollars)")
axes[0, 1].tick_params(axis="x", rotation=35)

for bar, value in zip(bars, action_values):
    axes[0, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"${value:.0f}M",
        ha="center",
        va="bottom",
    )

utilization_range = np.linspace(0.5, 1.0, 100)
congestion_penalties = [
    CapacityState(3.2, utilization, 0).get_congestion_penalty()
    for utilization in utilization_range
]
axes[1, 0].plot(
    utilization_range,
    congestion_penalties,
    color="blue",
    linewidth=2,
    label="Congestion penalty",
)
axes[1, 0].axvline(
    x=current_state.utilization_rate,
    color="red",
    linestyle="--",
    label=f"Initial utilization ({current_state.utilization_rate:.1%})",
)
axes[1, 0].axvline(
    x=0.8,
    color="orange",
    linestyle=":",
    label="Congestion threshold (80%)",
)
axes[1, 0].set_title("Daily Congestion Penalty")
axes[1, 0].set_xlabel("Utilization rate")
axes[1, 0].set_ylabel("Daily penalty (dollars)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

scenario_names = list(mdp.demand_scenarios.keys())
scenario_probabilities = [
    scenario["probability"]
    for scenario in mdp.demand_scenarios.values()
]
scenario_growth_rates = [
    scenario["growth_rate"]
    for scenario in mdp.demand_scenarios.values()
]
scenario_bars = axes[1, 1].bar(
    scenario_names,
    scenario_probabilities,
    color=["skyblue", "lightgreen", "salmon"],
)
axes[1, 1].set_title("Demand-Growth Scenario Probabilities")
axes[1, 1].set_xlabel("Scenario")
axes[1, 1].set_ylabel("Probability")
axes[1, 1].set_ylim(0, max(scenario_probabilities) + 0.12)

for bar, growth_rate in zip(scenario_bars, scenario_growth_rates):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"Growth: {growth_rate:.1%}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()
print("The four panels summarize the computed MDP quantities and model inputs.")

In [ ]:
# Summarize the initial state, its selected action, and the main interpretation limits.
print("\n" + "=" * 72)
print("PORT CAPACITY EXPANSION MDP: POLICY SUMMARY")
print("=" * 72)
print("\nINITIAL STATE")
print(f"Capacity: {current_state.capacity_mteu:.1f} million TEU")
print(f"Demand: {current_demand:.1f} million TEU")
print(f"Utilization: {current_state.utilization_rate:.1%}")
print(
    "Daily congestion penalty: "
    f"${current_state.get_congestion_penalty():,.0f}"
)

print("\nACTION SELECTED BY THE DISCOUNTED MDP")
print(f"Action: {optimal_action.name}")
print(f"Capacity increase: {optimal_action.capacity_increase:.0%}")
print(f"Period investment cost: ${optimal_action.cost_millions} million")
print(f"Estimated discounted value: ${optimal_value:.1f} million")

print("\nINTERPRETATION")
print(
    "The selected action maximizes the estimated Bellman value for the "
    "initial state under the specified rewards, transitions, and discount factor."
)
print(
    "The result depends on the finite capacity-demand grid and its nearest-level "
    "projection, as well as the assumption that action costs recur whenever an "
    "action is selected."
)

print("\nUNCERTAINTY AND MODEL SCOPE")
print("Demand growth scenarios:")
for scenario_name, scenario in mdp.demand_scenarios.items():
    print(
        f"  {scenario_name.title()}: "
        f"growth {scenario['growth_rate']:.1%}, "
        f"probability {scenario['probability']:.0%}"
    )
print(
    "Construction delays, cost overruns, financing constraints, regulation, "
    "and competitor responses are outside this model."
)
print("=" * 72)